# 02 – Nettoyage et fusion des deux jeux de données

Ce notebook applique les décisions issues de `01_eda.ipynb` et produit le **dataset consolidé** `data/processed/crop_yield_consolidated.csv`, source de vérité pour l'entraînement.

**Choix validés**

| Décision | Choix retenu |
|---|---|
| Stratégie de fusion | **Option A** : les observations FAO forment la base (elles départagent les cultures) ; le jeu Kaggle fournit l'**effet des pratiques** (engrais, irrigation), joint **par culture** |
| Cultures couvertes | les **10 cultures FAO** ; les 6 absentes de Kaggle reçoivent le gain global |
| Pesticides | valeur brute conservée ; la transformation (brute / log / exclusion) sera **mesurée** dans MLflow à l'étape 2 |

Toute la logique vit dans `src/agritech_crop_reco/data/` (`clean.py`, `merge.py`), testée par pytest. Ce notebook l'**appelle** et en montre les effets : le pipeline d'entraînement réutilisera exactement le même code (`python -m agritech_crop_reco.data.merge`).

In [1]:
import hashlib

import pandas as pd
import seaborn as sns

from agritech_crop_reco.config import PROJECT_ROOT, load_settings
from agritech_crop_reco.data.clean import (
    GLOBAL,
    KAGGLE_TARGET,
    clean_crop_yield,
    clean_fao,
    compute_practice_gains,
)
from agritech_crop_reco.data.load import load_crop_yield, load_fao_yield_df
from agritech_crop_reco.data.merge import (
    CONSOLIDATED_FILE,
    build_consolidated,
    merge_datasets,
    validate_consolidated,
)

settings = load_settings()
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", "{:,.3f}".format)

## 1. Agriculture Crop Yield : nettoyage

Seule anomalie relevée en EDA : des rendements ≤ 0 t/ha, impossibles physiquement.

In [2]:
crop_raw = load_crop_yield()
crop = clean_crop_yield(crop_raw)
print(
    f"{len(crop_raw):,} -> {len(crop):,} lignes "
    f"({len(crop_raw) - len(crop)} supprimées, "
    f"{(len(crop_raw) - len(crop)) / len(crop_raw):.3%} du jeu)"
)

1,000,000 -> 999,769 lignes (231 supprimées, 0.023% du jeu)


## 2. Mesure de l'effet des pratiques

### 2.1 Les effets s'additionnent-ils ou se multiplient-ils ?

Avant de choisir comment résumer l'effet de l'engrais et de l'irrigation, on regarde le rendement moyen pour les **4 combinaisons** de pratiques, et on compare la combinaison « les deux » à ce que prédirait chaque hypothèse.

In [3]:
combos = crop.groupby(["Fertilizer_Used", "Irrigation_Used"])[KAGGLE_TARGET].mean().unstack()
combos.index.name, combos.columns.name = "Engrais", "Irrigation"
display(combos)

none, fert, irr = combos.loc[False, False], combos.loc[True, False], combos.loc[False, True]
pd.Series(
    {
        "observé (engrais + irrigation)": combos.loc[True, True],
        "hypothèse additive : E + I - aucun": fert + irr - none,
        "hypothèse multiplicative : E × I / aucun": fert * irr / none,
    }
).to_frame("rendement moyen (t/ha)")

Irrigation,False,True
Engrais,,
False,3.306,4.499
True,4.800,5.999


,rendement moyen (t/ha)
observé (engrais + irrigation),5.999
hypothèse additive : E + I - aucun,5.993
hypothèse multiplicative : E × I / aucun,6.532


**Lecture.** L'hypothèse **additive** reproduit la combinaison observée à quelques millièmes près ; l'hypothèse multiplicative la surestime d'environ 0,5 t/ha. Dans ce jeu, chaque pratique **ajoute** un nombre fixe de tonnes, indépendamment de l'autre.

On ne peut pas pour autant transférer ces tonnes telles quelles aux cultures FAO : +1,5 t/ha doublerait le rendement du soja (≈ 1,6 t/ha) mais ne pèserait que 10 % sur la pomme de terre (≈ 16 t/ha). On exprime donc chaque effet **en pourcentage du rendement obtenu sans aucune pratique**, et on additionne les pourcentages :

$$\text{rendement}(E, I) = \text{rendement}_{\text{sans pratique}} \times (1 + g_E \cdot E + g_I \cdot I)$$

où $E$ et $I$ valent 0 ou 1. Les gains $g_E$ et $g_I$ sont estimés par une régression linéaire `rendement ~ engrais + irrigation` : $g = \text{coefficient} / \text{constante}$. Cette forme reproduit exactement les 4 combinaisons de Kaggle tout en s'adaptant à l'échelle de chaque culture.

### 2.2 Gains par culture

In [4]:
gains = compute_practice_gains(crop)
display(gains.set_index("crop").style.format("{:+.1%}"))

per_crop = gains[gains["crop"] != GLOBAL]
spread = per_crop[["fertilizer_gain", "irrigation_gain"]].agg(lambda s: s.max() - s.min())
print(
    f"écart entre cultures : engrais {spread['fertilizer_gain']:.2%}, "
    f"irrigation {spread['irrigation_gain']:.2%}"
)

,fertilizer_gain,irrigation_gain
crop,,
Barley,+45.8%,+36.1%
Cotton,+45.1%,+36.2%
Maize,+45.1%,+36.4%
Rice,+45.3%,+36.3%
Soybean,+45.2%,+36.5%
Wheat,+45.2%,+35.8%
__global__,+45.3%,+36.2%


écart entre cultures : engrais 0.67%, irrigation 0.73%


In [5]:
# Contrôle : la formule reproduit-elle les 4 combinaisons observées ?
g = gains.set_index("crop").loc[GLOBAL]
base = combos.loc[False, False]
check = pd.DataFrame(
    [
        {
            "engrais": e,
            "irrigation": i,
            "observé": combos.loc[e, i],
            "formule": base * (1 + g["fertilizer_gain"] * e + g["irrigation_gain"] * i),
        }
        for e in (False, True)
        for i in (False, True)
    ]
)
check["écart (t/ha)"] = check["formule"] - check["observé"]
check

,engrais,irrigation,observé,formule,écart (t/ha)
0,False,False,3.306,3.306,0.000
1,False,True,4.499,4.503,0.003
2,True,False,4.800,4.803,0.004
3,True,True,5.999,6.000,0.001


**Lecture.** Engrais ≈ **+45 %** et irrigation ≈ **+36 %** par rapport à une parcelle sans aucune pratique ; les deux ensemble ≈ +81 %. L'écart entre cultures est inférieur à 1 point : utiliser le **gain global** pour les 6 cultures FAO absentes de Kaggle est cohérent avec les données. La formule reproduit les 4 combinaisons à moins de 0,01 t/ha près.

⚠️ Ces gains décrivent **le simulateur Kaggle**, pas une mesure agronomique de terrain. Ce sont des ordres de grandeur plausibles (la littérature rapporte des gains de rendement importants avec fertilisation et irrigation), mais ils doivent être présentés comme une **hypothèse de travail** dans le rapport.

## 3. Crop Yield Prediction Dataset : nettoyage

Trois opérations (cf. EDA) :
1. **une ligne par pays × culture × année** : les copies dues aux multiples stations de température sont fusionnées en prenant la **moyenne des températures** (la fonction lève une erreur si une autre colonne variait au sein d'une clé) ;
2. **conversion** du rendement de hg/ha en t/ha ;
3. **harmonisation** des noms : `Rice, paddy` → `Rice`, `Soybeans` → `Soybean`.

In [6]:
fao_raw = load_fao_yield_df()
fao = clean_fao(fao_raw)
print(f"{len(fao_raw):,} -> {len(fao):,} lignes ({len(fao_raw) - len(fao):,} copies fusionnées)")

impact = pd.DataFrame(
    {
        "avant": {
            "part de l'Inde dans le jeu": (fao_raw["Area"] == "India").mean(),
            "pesticides moyens (t)": fao_raw["pesticides_tonnes"].mean(),
            "rendement moyen (t/ha)": fao_raw["hg/ha_yield"].mean() / 10_000,
        },
        "après": {
            "part de l'Inde dans le jeu": (fao["country"] == "India").mean(),
            "pesticides moyens (t)": fao["pesticides_tonnes"].mean(),
            "rendement moyen (t/ha)": fao["yield_t_ha"].mean(),
        },
    }
)
display(impact)
fao.head()

28,242 -> 13,130 lignes (15,112 copies fusionnées)


,avant,après
part de l'Inde dans le jeu,0.143,0.014
pesticides moyens (t),"37,076.909","14,838.698"
rendement moyen (t/ha),7.705,7.097


,country,crop,year,rainfall_mm,pesticides_tonnes,avg_temp_c,yield_t_ha
0,Albania,Maize,1990,"1,485.000",121.000,16.370,3.661
1,Albania,Maize,1991,"1,485.000",121.000,15.360,2.907
2,Albania,Maize,1992,"1,485.000",121.000,16.060,2.488
3,Albania,Maize,1993,"1,485.000",121.000,16.050,2.418
4,Albania,Maize,1994,"1,485.000",201.000,16.960,2.585


**Lecture.** Le dédoublonnage corrige un biais réel : l'Inde passait de **14 %** des lignes à environ **1 %**, sa part « juste » parmi 101 pays. Les statistiques descriptives changent sensiblement (les pesticides moyens sont divisés par plus de deux, car les pays les plus dupliqués sont de gros consommateurs). Sans cette correction, le modèle aurait surtout appris les grands pays, et une même observation aurait pu se retrouver à la fois en entraînement et en test.

## 4. Fusion

Jointure **gauche** des observations FAO avec les gains, **sur la culture** (`many_to_one` vérifié par pandas). Les cultures sans gain propre reçoivent le gain global ; la colonne `gain_source` indique l'origine (`crop` ou `global`).

In [7]:
consolidated = merge_datasets(fao, gains)
summary = (
    consolidated.groupby(["crop", "gain_source"])
    .agg(
        lignes=("yield_t_ha", "size"),
        pays=("country", "nunique"),
        rendement_median=("yield_t_ha", "median"),
        gain_engrais=("fertilizer_gain", "first"),
        gain_irrigation=("irrigation_gain", "first"),
    )
    .sort_values("rendement_median", ascending=False)
)
summary

,,lignes,pays,rendement_median,gain_engrais,gain_irrigation
crop,gain_source,,,,,
Potatoes,global,2091,95,16.189,0.453,0.362
Cassava,global,941,41,10.000,0.453,0.362
Plantains and others,global,464,21,9.702,0.453,0.362
Yams,global,454,21,9.240,0.453,0.362
Sweet potatoes,global,1163,53,8.750,0.453,0.362
Rice,crop,1502,68,3.355,0.453,0.363
Maize,crop,2028,92,2.449,0.451,0.364
Wheat,crop,1810,82,2.432,0.452,0.358
Soybean,crop,1242,58,1.620,0.452,0.365


## 5. Contrôles qualité et écriture

`validate_consolidated` bloque l'écriture si : doublon de clé, valeur manquante, rendement ≤ 0 ou gain hors de [0, 1]. On vérifie aussi que la fonction du pipeline (`build_consolidated`) produit **exactement** le même tableau que les étapes ci-dessus.

In [8]:
consolidated = consolidated.sort_values(["country", "crop", "year"]).reset_index(drop=True)
validate_consolidated(consolidated)

from_pipeline = build_consolidated()
pd.testing.assert_frame_equal(consolidated, from_pipeline)
print("Contrôles OK - notebook et pipeline produisent le même dataset")

out = settings.processed_dir / CONSOLIDATED_FILE
out.parent.mkdir(parents=True, exist_ok=True)
consolidated.to_csv(out, index=False)
digest = hashlib.sha256(out.read_bytes()).hexdigest()
rel_path = out.relative_to(PROJECT_ROOT)
print(f"{len(consolidated):,} lignes × {consolidated.shape[1]} colonnes -> {rel_path}")
print(f"sha256 : {digest}")

Contrôles OK - notebook et pipeline produisent le même dataset
13,130 lignes × 10 colonnes -> data/processed/crop_yield_consolidated.csv
sha256 : 35a7bf9a0d5510030aeb31459c0af4ec746b642761bd6ec458366b9df17a18a5


In [9]:
dictionary = pd.DataFrame(
    [
        ("country", "texte", "pays (FAO)", "clé"),
        ("crop", "texte", "culture, noms harmonisés avec Kaggle", "clé / variable explicative"),
        ("year", "entier", "année d'observation (1990–2013)", "clé"),
        ("rainfall_mm", "réel", "pluie moyenne annuelle du pays (mm)", "variable explicative"),
        (
            "pesticides_tonnes",
            "réel",
            "pesticides du pays (t de matière active)",
            "variable explicative",
        ),
        (
            "avg_temp_c",
            "réel",
            "température moyenne annuelle (°C), moyenne des stations",
            "variable explicative",
        ),
        ("yield_t_ha", "réel", "rendement (t/ha), converti depuis hg/ha", "cible"),
        ("fertilizer_gain", "réel", "gain relatif dû à l'engrais (Kaggle)", "ajustement pratiques"),
        (
            "irrigation_gain",
            "réel",
            "gain relatif dû à l'irrigation (Kaggle)",
            "ajustement pratiques",
        ),
        (
            "gain_source",
            "texte",
            "origine du gain : crop (mesuré sur la culture) ou global",
            "traçabilité",
        ),
    ],
    columns=["colonne", "type", "description", "rôle"],
)
assert list(dictionary["colonne"]) == list(consolidated.columns)
dictionary

,colonne,type,description,rôle
0,country,texte,pays (FAO),clé
1,crop,texte,"culture, noms harmonisés avec Kaggle",clé / variable explicative
2,year,entier,année d'observation (1990–2013),clé
3,rainfall_mm,réel,pluie moyenne annuelle du pays (mm),variable explicative
4,pesticides_tonnes,réel,pesticides du pays (t de matière active),variable explicative
5,avg_temp_c,réel,"température moyenne annuelle (°C), moyenne des...",variable explicative
6,yield_t_ha,réel,"rendement (t/ha), converti depuis hg/ha",cible
7,fertilizer_gain,réel,gain relatif dû à l'engrais (Kaggle),ajustement pratiques
8,irrigation_gain,réel,gain relatif dû à l'irrigation (Kaggle),ajustement pratiques
9,gain_source,texte,origine du gain : crop (mesuré sur la culture)...,traçabilité


## 6. Point ouvert pour l'étape 3 : à quoi correspond le rendement FAO ?

Le rendement FAO est une **moyenne nationale**, qui mélange déjà des parcelles avec et sans engrais ou irrigation. Ce n'est donc pas un rendement « sans aucune pratique ». Deux façons d'appliquer les gains au moment de la prédiction :

| Hypothèse | Facteur appliqué à la prédiction FAO | Effet |
|---|---|---|
| (a) FAO = parcelle sans pratique | $1 + g_E E + g_I I$ | simple, mais **surestime** : « engrais + irrigation » donnerait +81 % au-dessus d'une moyenne qui en contient déjà |
| (b) FAO = pratiques moyennes (recommandée) | $\dfrac{1 + g_E E + g_I I}{1 + p_E g_E + p_I g_I}$ | une parcelle « moyenne » garde la prédiction FAO ; avec les deux pratiques ≈ +29 %, sans aucune ≈ −29 % (avec $p_E = p_I = 0{,}5$, proportions du jeu Kaggle) |

Les colonnes du CSV ($g_E$, $g_I$) permettent les deux : le choix se fera à l'étape 3 (`recommend.py`), sans refaire la fusion.

## 7. Résumé des choix (pour le rapport métier)

**Pourquoi cette fusion ?** Les deux jeux ne partagent ni pays, ni année, ni région : la **culture** est la seule clé commune. Surtout, ils n'apportent pas la même information :
- le jeu **FAO** (101 pays, 1990–2013) montre que **la culture est le premier facteur de rendement**, ce qui est indispensable pour recommander une culture ;
- le jeu **Kaggle** (1 million de parcelles simulées) n'a aucun effet culture, mais **isole proprement l'effet de l'engrais et de l'irrigation**, absents de la FAO.

On combine donc les forces des deux : le **modèle de rendement** apprend sur les données FAO, et l'effet des **pratiques** mesuré sur Kaggle ajuste ensuite sa prédiction.

**Ce que le nettoyage a corrigé**
- 231 parcelles Kaggle au rendement négatif (artefact du simulateur) supprimées, soit 0,02 % du jeu ;
- 15 112 lignes FAO **en double** (une copie par station météo du pays) fusionnées : 28 242 → **13 130 observations réelles** ; sans cela, l'Inde pesait 14 % du jeu au lieu de 1 % ;
- unités harmonisées (t/ha) et noms de cultures alignés.

**Chiffres clés pour un non-spécialiste**
- Utiliser de l'**engrais** augmente le rendement d'environ **45 %**, l'**irrigation** d'environ **36 %** (par rapport à une parcelle sans aucune des deux), et ces gains se cumulent ; ils sont identiques quelle que soit la culture dans les données disponibles.
- Les rendements varient d'un facteur 10 selon la culture (≈ 16 t/ha pour la pomme de terre contre ≈ 1,2 t/ha pour le sorgho).

**Limites assumées**
- Les gains viennent d'un jeu **simulé** : ce sont des ordres de grandeur, pas des mesures de terrain.
- Ils sont appliqués à 6 cultures absentes de Kaggle, en s'appuyant sur le constat qu'ils ne varient pas d'une culture à l'autre.
- Les variables FAO sont **nationales et annuelles** (pluie, température, pesticides) : le modèle décrit des conditions de pays, pas de parcelle.